# PUMA Exploration 8 — staged batch runner

- Current active data scope: `QUICK_BENCHMARK` only.
- Stages without a manual decision are grouped in loops.
- Decision/final/full-data cells stay disabled with `""" ... """` until intentionally enabled.
- Completed experiment stages are reused automatically; rerunning the notebook does **not** retrain them.


In [ ]:
# 1. MOUNT DRIVE + PROJECT PATH
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys

ROOT = Path('/content/drive/MyDrive/Research/PUMA/Exploration_8_Diversity_Aware_Wider_Field_and_Boundary_Optimization')
PATHS_JSON = ROOT / 'INPUTS/QUICK_BENCHMARK/PATHS_COLAB.json'
sys.path.insert(0, str(ROOT / 'CODE/COMMON/src'))


In [ ]:
# 2. RUNNER
from puma_exploration8.workflow import run_stage, status, freeze_decision
from puma_exploration8.runtime import release_cuda

DEVICE = 'auto'
FINAL_SEED = 17

def run_many(stages, *, paths_json=PATHS_JSON):
    for stage in stages:
        print(f'\n=== {stage} ===')
        try:
            run_stage(
                ROOT,
                stage,
                paths_json=paths_json,
                device=DEVICE,
                seed=FINAL_SEED,
                force=False,  # completed experiment -> reuse/skip, never retrain
            )
        finally:
            release_cuda()

    return status(ROOT, paths_json=paths_json, seed=FINAL_SEED)


## 3. Active now — partial-data exploratory batch

This is the block to run now. It uses only `QUICK_BENCHMARK`.

`smoke → prepare → p8c → p8c_exposure → p8d` does not require a frozen wider/route/Tier-A choice beforehand, so these stages are batched together.


In [ ]:
# ACTIVE: run the first research block on QUICK_BENCHMARK only.
INITIAL_STAGES = [
    "smoke",
    "prepare",
    "p8c",
    "p8c_exposure",
    "p8d",
]

run_many(INITIAL_STAGES)


## 4. Decision gate — exposure policy

Review the P8-C exposure result first. This cell is intentionally disabled.
Remove the opening/closing triple quotes only when you are ready to freeze the choice.


In [ ]:

# DECISION: choose exactly one after reviewing P8-C exposure results.
EXPOSURE = "inverse" # "inverse" or "group"

if EXPOSURE not in {"inverse", "group"}:
    raise ValueError('Set EXPOSURE to "inverse" or "group" before freezing.')

freeze_decision(
    ROOT,
    "exposure",
    EXPOSURE,
    source="P8C",
    paths_json=PATHS_JSON or None,
)



## 5. Decision gate — wider-field representation

Review P8-D, then choose `gaussian192` or `annular192`. This cell is intentionally disabled.


In [ ]:

# DECISION: choose exactly one after reviewing P8-D.
SELECTED_WIDER = "gaussian192"  # "gaussian192" or "annular192"

if SELECTED_WIDER not in {"gaussian192", "annular192"}:
    raise ValueError('Set SELECTED_WIDER before freezing.')

freeze_decision(
    ROOT,
    "selected_wider",
    SELECTED_WIDER,
    source="P8D",
    paths_json=PATHS_JSON or None,
)



## 6. Run P8-E — only after wider/exposure choices are frozen

This stage is disabled for now so the notebook stops at the decision gate.


In [ ]:

run_many(["p8e"])



## 7. Decision gate — routing

Review P8-E, then choose route `A` or `C`. This cell is intentionally disabled.


In [ ]:

# DECISION: choose exactly one after reviewing P8-E.
SELECTED_ROUTE = "C"  # "A" or "C"

if SELECTED_ROUTE not in {"A", "C"}:
    raise ValueError('Set SELECTED_ROUTE before freezing.')

freeze_decision(
    ROOT,
    "selected_route",
    SELECTED_ROUTE,
    source="P8E",
    paths_json=PATHS_JSON or None,
)



## 8. Run P8-F — only after route is frozen

This stage is disabled for now.


In [ ]:

run_many(["p8f"])



## 9. Decision gate — Tier-A

Review P8-F, then explicitly choose whether Tier-A is retained. This cell is intentionally disabled.


In [ ]:

# DECISION: set True or False after reviewing P8-F.
USE_TIERA = False

if not isinstance(USE_TIERA, bool):
    raise ValueError("Set USE_TIERA to True or False before freezing.")

freeze_decision(
    ROOT,
    "tiera",
    USE_TIERA,
    source="P8F",
    paths_json=PATHS_JSON or None,
)



## 10. Remaining training/calibration batch

Once the scientific choices above are frozen, these stages no longer need new manual selections and can run as one loop.

Disabled for now.


In [ ]:

run_many([
    "p8g",
    "final_refit",
    "boundary",
])



## 11. Protected evaluation — keep separate

`VALIDATE` and `TEST` are intentionally not put into the normal training loop. Run VALIDATE first, inspect the promotion result, and only then run TEST if allowed.

Both cells are disabled for now.


In [ ]:

# Protected one-shot VALIDATE
run_many(["validate"])



In [ ]:

# Protected one-shot TEST — enable only after VALIDATE promotes the candidate.
run_many(["test"])



## 12. Full / one-click run — disabled

This convenience block is intentionally commented out. It assumes all required scientific decisions have already been frozen and is **not** the mode to use while you are still working on the partial benchmark.


In [ ]:
"""
# RUN ALL STAGES — DISABLED FOR NOW.
# Use only later, after the required decisions are already frozen.
ALL_STAGES = [
    "smoke",
    "prepare",
    "p8c",
    "p8c_exposure",
    "p8d",
    "p8e",
    "p8f",
    "p8g",
    "final_refit",
    "boundary",
    "validate",
    "test",
]

run_many(ALL_STAGES)
"""


## 13. Full-data mode — disabled

The notebook currently points to `INPUTS/QUICK_BENCHMARK/PATHS_COLAB.json`. Do not enable the block below until you intentionally switch to the full dataset and have the correct full-data paths JSON.


In [ ]:
"""
# FULL DATA MODE — DISABLED FOR NOW.
# Set this only when the full-data PATHS JSON is ready.
FULL_PATHS_JSON = None

if not FULL_PATHS_JSON:
    raise ValueError("Set FULL_PATHS_JSON before enabling full-data mode.")

# Example pattern after all required decisions are frozen for that output root:
# run_many(["prepare", "p8c", "p8c_exposure", "p8d"], paths_json=FULL_PATHS_JSON)
"""
